**This notebook is to create a table using system id**

In [0]:
#import libraries
import yaml

In [0]:
#declaring varibales
yaml_path = "abfss://inbound@adlsstorageaccount5858.dfs.core.windows.net/config/yaml/control/etl_run_log.yaml"
# yaml_path = yaml_base_path+yaml_file_name

In [0]:
yaml_files = dbutils.fs.head(yaml_path, 10000)
yaml_config =  yaml.safe_load(yaml_files)
#read yaml file
# with open(yaml_path, "r") as yaml_files:
#     yaml_config = yaml.safe_load(yaml_files)
# print(yaml_config)
# print(yaml_config["table"])
table_name = yaml_config["table"]["name"]
table_schema_name = f"customer_billing.{yaml_config['table']['schema_name']}"
table_columns = yaml_config["table"]["columns"]
table_column_name = ",\n ".join([f'{table_column["name"]} {table_column["data_type"]}' for table_column in table_columns])
# print(table_column_name)

In [0]:
#Create schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {table_schema_name}")

#Prepare CREATE TABLE statement
create_table_sql = f"""CREATE TABLE IF NOT EXISTS {table_schema_name}.{table_name} ({table_column_name}) USING DELTA"""

#Prepare complete table name
full_table_name = f"{table_schema_name}.{table_name}"

# Check whether table already exists
if spark.catalog.tableExists(full_table_name):
    print(f"Table already exists: {full_table_name}")
else:
    # Create table only when it does not exist
    spark.sql(create_table_sql)
    print(f"Table created successfully: {full_table_name}")

In [0]:
# create_table_sql = f"""
# create table if not exists {table_schema_name}.{table_name}({table_column_name})
# """
# # print(create_table_sql)
# spark.sql(create_table_sql)